# Split Train and Validation Dataset

This notebook creates a reproducible image-level train/validation split for the prepared VinBig 1024 x 1024 detection annotations. It does not rebalance by deleting samples. Instead, it preserves all annotations and writes split-aware CSV files that the EfficientDet-D4 training notebook can consume.

The split is image-level, not annotation-row-level, so all boxes for the same image stay in the same split. The validation selection is class-aware and gives special attention to the rare classes: Consolidation, Atelectasis, and Pneumothorax.

## Setup

Import path helpers, random utilities, and pandas for CSV processing.

In [ ]:
# Notebook guide: shared imports for creating a reproducible image-level split.
# Pandas handles annotation tables; `random` controls deterministic image selection.

from pathlib import Path
import random

import pandas as pd

## Configuration

Define the source annotation CSV, output CSV names, validation size, random seed, and rare classes to monitor during the split.

In [ ]:
# Notebook guide: configure the split source and outputs.
# All output files start with `04_` to match this preparation step.

PROCESSED_DIR = Path("../data/processed")
SOURCE_CSV = PROCESSED_DIR / "02_vinbig_transformed_bounding_boxes_1024.csv"

SPLIT_CSV = PROCESSED_DIR / "04_vinbig_train_val_split_1024.csv"
TRAIN_CSV = PROCESSED_DIR / "04_vinbig_train_split_1024.csv"
VAL_CSV = PROCESSED_DIR / "04_vinbig_val_split_1024.csv"

VAL_FRACTION = 0.15
SEED = 42
RARE_CLASS_NAMES = {"Consolidation", "Atelectasis", "Pneumothorax"}

## Load Annotations

Read the prepared VinBig annotations and verify the expected detection columns are available before creating splits.

In [ ]:
# Notebook guide: load the prepared 1024-space annotations and validate the schema.
# The split is created from image IDs so each image appears in exactly one split.

df = pd.read_csv(SOURCE_CSV)
required_columns = {"image_id", "class_name", "target_class_id", "x_min", "y_min", "x_max", "y_max"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df["image_id"] = df["image_id"].astype(str)
df["target_class_id"] = df["target_class_id"].astype(int)

print(f"Annotation rows: {len(df):,}")
print(f"Unique images: {df['image_id'].nunique():,}")
display(df.groupby("class_name").size().sort_values())

## Create Class-Aware Image Split

Select validation images greedily using class deficits. At each step, the next validation image is the one that best fills the current per-class validation target. This keeps the split reproducible and helps rare classes appear in validation without removing any training samples from the source data.

In [ ]:
# Notebook guide: create an image-level split that tries to preserve class coverage.
# Rare classes receive extra priority while selecting validation images.

rng = random.Random(SEED)
image_groups = {image_id: group for image_id, group in df.groupby("image_id")}
image_ids = sorted(image_groups)
target_val_images = max(1, round(len(image_ids) * VAL_FRACTION))

class_counts = df.groupby("class_name").size().to_dict()
target_val_counts = {label: max(1, round(count * VAL_FRACTION)) for label, count in class_counts.items()}
val_counts = {label: 0 for label in class_counts}

remaining = set(image_ids)
val_ids = set()

def image_class_counts(image_id: str) -> dict[str, int]:
    return image_groups[image_id].groupby("class_name").size().to_dict()


def image_score(image_id: str) -> float:
    score = 0.0
    for label, count in image_class_counts(image_id).items():
        deficit = max(0, target_val_counts[label] - val_counts[label])
        if deficit == 0:
            continue
        rare_boost = 2.0 if label in RARE_CLASS_NAMES else 1.0
        score += rare_boost * min(count, deficit) / target_val_counts[label]
    return score + rng.random() * 1e-6


while remaining and len(val_ids) < target_val_images:
    selected = max(remaining, key=image_score)
    val_ids.add(selected)
    remaining.remove(selected)
    for label, count in image_class_counts(selected).items():
        val_counts[label] += count

train_ids = set(image_ids) - val_ids

print(f"Train images: {len(train_ids):,}")
print(f"Val images:   {len(val_ids):,}")
print(f"Val image fraction: {len(val_ids) / len(image_ids):.3f}")

## Verify Split Distribution

Attach the `split` column, then compare annotation counts per class in train and validation. The validation split should include the rare classes so model evaluation can track them.

In [ ]:
# Notebook guide: add the split label and inspect class counts before writing files.
# This check helps catch rare classes that accidentally disappeared from validation.

split_df = df.copy()
split_df["split"] = split_df["image_id"].map(lambda image_id: "val" if image_id in val_ids else "train")

summary = (
    split_df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda table: table.sum(axis=1))
)
summary["val_fraction"] = (summary.get("val", 0) / summary["total"]).round(3)
display(summary.sort_values("total"))

rare_summary = summary.loc[summary.index.intersection(RARE_CLASS_NAMES)]
if (rare_summary.get("val", 0) == 0).any():
    missing = rare_summary[rare_summary.get("val", 0) == 0].index.tolist()
    raise ValueError(f"Rare classes missing from validation split: {missing}")

## Write Split CSVs

Write one combined split-aware CSV plus separate train and validation CSVs. The training notebook reads the combined file by default.

In [ ]:
# Notebook guide: save the split outputs under `data/processed` with the `04_` prefix.
# The combined CSV is the source of truth; separate train/val CSVs are convenience exports.

train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "val"].copy()

split_df.to_csv(SPLIT_CSV, index=False)
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)

print(f"Wrote combined split CSV: {SPLIT_CSV}")
print(f"Wrote train CSV:          {TRAIN_CSV}")
print(f"Wrote val CSV:            {VAL_CSV}")